In [16]:
# !pip install missingno
# !pip install geopy

In [17]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
AirbnbBerlin_df = pd.read_csv('/content/drive/My Drive/Airbnb/Airbnb Berlin.csv', index_col=0)

In [20]:
AirbnbBerlin_df.groupby(['Listing ID', 'Comments']).size().sort_values(ascending=False)

Listing ID  Comments                                                                                                                                                                                                                                                      
3838061     The host canceled this reservation 2 days before arrival. This is an automated posting.                                                                                                                                                                           5
3227823     The host canceled this reservation the day before arrival. This is an automated posting.                                                                                                                                                                          4
3298027     The host canceled this reservation 2 days before arrival. This is an automated posting.                                                                                                                                                                           4
237038      nice stay                                                                                                                                                                                                                                                         4
1626236     The host canceled this reservation 2 days before arrival. This is an automated posting.                                                                                                                                                                           4
                                                                                                                                                                                                                                                                             ..
5158546     I had the pleasure of staying in this wonderful place while visiting Berlin. Dagmar is a great host and communication is very pleasant. The house is located in a very nice neighbourbood with everything nearby. The French restaurant around the corner is s    1
            I had good hopes, reading previous comments, but it was better than expected anyway, I will definitely go back at my next visit in Berlin. Lovely place, peace and quiet. Amazing breakfast! Personal way of treating guests... 10 out of 10!                     1
            Had a very pleasant stay in Dagmar's place.\nThe room is as described and shown in the photo. \nThe place is clean, quiet, the breakfast is good, with plenty of choices and large quantities.  \nDagmar is available and very welcoming.\n\nThe neighborhood     1
            Great stay in a quiet but beautiful area of Berlin. Great square near by for a tasty glass of riesling at a good price and a 3 minute walk to the Ubahn to see more of Berlin. Excellent value for the price paid.                                                1
            Nette kleine Pension, mit sehr viel Liebe eingerichtet (Kronleuchter, Spiegel, Bilder, PlÃ¼sch...), ruhige schÃ¶ne Gegend, sehr reichhaltiges FrÃ¼hstÃ¼cksbÃ¼ffet.                                                                                                1
Length: 452312, dtype: int64

In [21]:
df = AirbnbBerlin_df.copy()

# Data Preparation

In [22]:
# Display shape
df.shape

(456961, 46)

## Aggregate dataset by Listing ID

Aggregate the dataset by 'Listing ID'
- For numerical columns, we'll compute the mean
- For categorical columns, we'll take the first/or last value (assuming consistency)
- Clean text From pancutations or undesired characters

In [23]:
# Clean Text: Perform text cleaning, remove currency symbols & commas
df['Price'] = df['Price'].replace('[\$,]', '', regex=True).astype(float)
df['Host Response Rate'] = df['Host Response Rate'].replace('[\%,]', '', regex=True).astype(float)
df['Host Response Rate'] = df['Host Response Rate'].astype(float)

# Fix Postal Code incorrect values, remove '\n' and other irrelevant text
df['Postal Code'] = df['Postal Code'].astype(str).str[:5]

drop redundant columns

In [24]:
# df = df.drop(columns=['Review ID', 'Reviewer ID', 'Reviewer Name', 'Listing URL','Listing Name', 'Host ID',
#                       'Host URL', 'Host Name', 'City', 'Country Code', 'Country', 'Last Review', 'First Review',])

In [25]:
# Aggregate the dataset by 'Listing ID'
# For numerical columns, we'll compute the mean
# For categorical columns, we'll take the first value (assuming consistency)

categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['number']).columns.difference(['Listing ID'])

categorical_cols = categorical_cols.append(pd.Index(['Accomodates', 'Bedrooms', 'Beds', 'Guests Included','Min Nights','Reviews']))
numerical_cols = numerical_cols.difference(categorical_cols)

aggregated_df = df.groupby('Listing ID').agg({**{col: 'mean' for col in numerical_cols},
                                              **{col: 'first' for col in categorical_cols}})

# aggregated_df = df.groupby('Listing ID').agg({**{col: 'mean' for col in numerical_cols},
#                                               **{col: lambda x: x.mode().iloc[0] if not x.mode().empty else None for col in categorical_cols}})

df = aggregated_df.reset_index()

## Insepction

In [26]:
# Display shape
df.shape

(23536, 46)

In [27]:
# Display head(2) of remaining "df"
df.head(2)

,Listing ID,Accuracy Rating,Bathrooms,Checkin Rating,Cleanliness Rating,Communication Rating,Host ID,Host Response Rate,Latitude,Location Rating,Longitude,Overall Rating,Price,Review ID,Reviewer ID,Square Feet,Value Rating,review_date,Reviewer Name,Comments,Listing URL,Listing Name,Host URL,Host Name,Host Since,Host Response Time,Is Superhost,neighbourhood,Neighborhood Group,City,Postal Code,Country Code,Country,Is Exact Location,Property Type,Room Type,First Review,Last Review,Instant Bookable,Business Travel Ready,Accomodates,Bedrooms,Beds,Guests Included,Min Nights,Reviews
0,2695,10.0,1.0,10.0,10.0,10.0,2986.0,50.0,52.54851,9.0,13.40455,100.0,17.0,3.303867e+08,4.097753e+07,NaN,10.0,07-04-18,Jason,I really enjoyed staying at Micha and Nadja's ...,https://www.airbnb.com/rooms/2695,Prenzlauer Berg close to Mauerpark,https://www.airbnb.com/users/show/2986,Michael,09-16-08,within a day,f,Prenzlauer Berg,Pankow,Berlin,10437,DE,Germany,t,Apartment,Private room,07-04-18,04-21-19,f,f,2,1.0,1.0,1,2,7
1,3176,9.0,1.0,9.0,9.0,9.0,3718.0,50.0,52.53500,10.0,13.41758,92.0,90.0,3.089398e+07,1.404133e+07,720.0,9.0,06-20-09,Milan,"excellent stay, i would highly recommend it. a...",https://www.airbnb.com/rooms/3176,Fabulous Flat in great Location,https://www.airbnb.com/users/show/3718,Britta,10-19-08,within a day,f,Prenzlauer Berg,Pankow,Berlin,10405,DE,Germany,t,Apartment,Entire home/apt,06-20-09,10-29-18,f,f,4,1.0,2.0,2,62,144


On first sight, most features appear relatively informative and well-structured.

Also, various numerical features are currently stored as objects and need to be transformed (e.g. Price, Host Response Rate, First Review, ...). Additionally, there are actually quite a few columns with missing values.

I notice that reviewer's comments were stored as a string, which needs to be reviewed later on.

At this point point, will move the comments column to a different dataset and store in sparated CSV file comments.csv

In [28]:
df_comments = df[['Listing ID', 'Comments']].copy()
df_comments.to_csv('/content/drive/My Drive/Airbnb/comments.csv')

Now we saved comment into different dataset, will drop it with other redundant columns

In [29]:
df = df.drop(columns=['Listing ID', 'Review ID', 'Reviewer ID', 'Reviewer Name', 'Listing URL','Listing Name', 'Host ID',
                      'Host URL', 'Host Name', 'City', 'Country Code', 'Country', 'Last Review', 'First Review',])

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23536 entries, 0 to 23535
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Accuracy Rating        18888 non-null  float64
 1   Bathrooms              23507 non-null  float64
 2   Checkin Rating         18870 non-null  float64
 3   Cleanliness Rating     18892 non-null  float64
 4   Communication Rating   18886 non-null  float64
 5   Host Response Rate     13046 non-null  float64
 6   Latitude               23536 non-null  float64
 7   Location Rating        18871 non-null  float64
 8   Longitude              23536 non-null  float64
 9   Overall Rating         18914 non-null  float64
 10  Price                  23536 non-null  float64
 11  Square Feet            425 non-null    float64
 12  Value Rating           18868 non-null  float64
 13  review_date            19380 non-null  object 
 14  Comments               19378 non-null  object 
 15  Ho

As expected, various numerical features are currently stored as objects and need to be transformed (e.g. Price, Host Response Rate, First Review, ...). Additionally, there are actually quite a few columns with missing values.

In [31]:
# categorical_cols
df.select_dtypes(include=['object']).columns

Index(['review_date', 'Comments', 'Host Since', 'Host Response Time',
       'Is Superhost', 'neighbourhood', 'Neighborhood Group', 'Postal Code',
       'Is Exact Location', 'Property Type', 'Room Type', 'Instant Bookable',
       'Business Travel Ready'],
      dtype='object')

As expected, various numerical features are currently stored as objects and need to be transformed (e.g. Price, Host Response Rate, First Review, ...). Additionally, there are actually quite a few columns with missing values.

In [32]:
# Describe data (summary)
df.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
Accuracy Rating,18888.0,9.68,0.74,2.00,10.00,10.00,10.00,10.00
Bathrooms,23507.0,1.10,0.35,0.00,1.00,1.00,1.00,8.50
Checkin Rating,18870.0,9.73,0.70,2.00,10.00,10.00,10.00,10.00
Cleanliness Rating,18892.0,9.33,1.03,2.00,9.00,10.00,10.00,10.00
Communication Rating,18886.0,9.75,0.68,2.00,10.00,10.00,10.00,10.00
Host Response Rate,13046.0,91.84,19.45,0.00,95.00,100.00,100.00,100.00
Latitude,23536.0,52.51,0.03,52.35,52.49,52.51,52.53,52.65
Location Rating,18871.0,9.55,0.75,2.00,9.00,10.00,10.00,10.00
Longitude,23536.0,13.41,0.06,13.10,13.38,13.42,13.44,13.76
Overall Rating,18914.0,94.56,7.60,20.00,92.00,97.00,100.00,100.00


In [33]:
# Show maximum/minimum value for each numerical column
num_features = df.columns[df.dtypes!=object].difference(['Review ID', 'Reviewer ID', 'Listing ID', 'Host ID', 'Latitude', 'Longitude'])
df[num_features].describe().loc[['min','max']].T

,min,max
Accomodates,1.0,16.0
Accuracy Rating,2.0,10.0
Bathrooms,0.0,8.5
Bedrooms,0.0,10.0
Beds,0.0,22.0
Checkin Rating,2.0,10.0
Cleanliness Rating,2.0,10.0
Communication Rating,2.0,10.0
Guests Included,1.0,16.0
Host Response Rate,0.0,100.0


In [34]:
# plt.figure(figsize=(16, 2))
# sns.boxplot(df['Price'], orient='h')
# plt.show()

Several rows with unusually high values can be identified and may in some cases be dropped at a certain threshold during data handling and before aggrgation step. Some particular features include:

The description of price shows that 75% of the room only charged within 70€. But we can find the maximized price is extremely large - up to 9000€.

To exclude the outlinear in this dataset, we set the data limit of 600€.

It also shows that data has 0 price, that is not make sense. So we will exclude it too.

In [35]:
# df['Price'] = df['Price'].replace('[\$,]', '', regex=True).astype(float)
# df = df[(df['Price'] <= 600) & (df['Price'] > 0)]
# df['Price'].describe()

## Features Selection (pre-cleaning)

1. If a categorical column is not relevant to the analysis, we can remove it.
2. Listing URL, Listing Name, Host URL, Host Name: These are mostly unique to each listing, so not useful for category reduction
3. columns with 50% missing values will be dropped like 'Square Feet'

#### List missing values

In [36]:
# List missing values (pre-cleaning)
null_cols = df.columns[df.isnull().any(axis=0)] ## any columns with at lest one null value
df[null_cols].isna().sum().sort_values(ascending=False)

,0
Square Feet,23111
Host Response Time,10490
Host Response Rate,10490
Value Rating,4668
Checkin Rating,4666
Location Rating,4665
Communication Rating,4650
Accuracy Rating,4648
Cleanliness Rating,4644
Overall Rating,4622


Various features have a lot of missing values. In particular, there is an observable cut where many features have more than 4.500 missing values and the rest has less than 1.000. The former - except for review_scores - shall be removed, the latter imputed.

#### List unique entries

In [37]:
# List unique entries per column
df.nunique().sort_values(ascending=False)

,0
Comments,17600
Longitude,14851
Latitude,11821
Host Since,3085
review_date,2382
Reviews,332
Price,326
Postal Code,207
Square Feet,109
Min Nights,97


Three main insights from unique values:

- Some columns have only 1 value and can be dropped
- Some other columns have 2 values and appear to be true/false (i.e. can be recoded as 1/0)
- Certain columns have a high number of unique values, which can probably be clustered into a few relevant ones (e.g. cancellation_policy, property_type)

#### Conclusions (selection) - dropping redundant columns

1. If a categorical column is not relevant to the analysis, we can remove it.
2. Listing URL, Listing Name, Host URL, Host Name: These are mostly unique to each listing, so not useful for category reduction
3. **columns with more than 50% missing values will be dropped like 'Square Feet'** TPD
4. 'Comment' columns can be dropped as it was stored in separated dataset
5. 'Business Travel Ready' has one value and can be dropped

In [38]:
# drop the columns that is not helpful for prediction
df = df.drop(columns=['Business Travel Ready']) # 'Square Feet',

## Reduce Large Categories

1. Group Rare Categories: If a categorical column has many unique values, we can group infrequent categories into an "Other" category like 'Reviewer Name'.
2. Merge Similar Categories: If there are similar categories (e.g., different spellings or formats of the same category), we can merge them.
3. Binning: For numerical categories (like "Overall Rating" or "Accommodates"), we can create bins to reduce the number of unique values.

In [39]:
# # Fix Postal Code incorrect values, remove '\n' and other irrelevant text
# df['Postal Code'] = df['Postal Code'].astype(str).str[:5]

# 1. Clean Text: Perform text cleaning, remove currency symbols & commas
df['Price'] = df['Price'].replace('[\$,]', '', regex=True).astype(float)
df['Host Response Rate'] = df['Host Response Rate'].replace('[\%,]', '', regex=True).astype(float)
df['Host Response Rate'] = df['Host Response Rate'].astype(float)

# 2. Grouping neighbourhoods into Neighborhood Groups
# if 'Neighborhood Group' in df.columns:
#   neighbourhood_mapping = df.groupby('neighbourhood')['Neighborhood Group'].first()
#   df['Neighbourhood Group Reduced'] = df['neighbourhood'].map(neighbourhood_mapping)

# 3. Reducing Property Types
property_mapping = {
    "Villa": "Vacation Rental",
    "Cottage": "Vacation Rental",
    "Bungalow": "Vacation Rental",
    "Cabin": "Vacation Rental",
    "Tiny house": "Vacation Rental",
    "Earth house": "Vacation Rental",
    "Treehouse": "Vacation Rental",
    "Hut": "Vacation Rental",
    "Barn": "Vacation Rental",
    "Houseboat": "Boats & Houseboats",
    "Boat": "Boats & Houseboats",
    "Camper/RV": "Mobile/Alternative Lodging",
    "Cave": "Mobile/Alternative Lodging",
    "Pension (South Korea)": "Mobile/Alternative Lodging",
    "Casa particular (Cuba)": "Mobile/Alternative Lodging",
}

# Apply mapping and assign 'Other' to rare categories
top_property_types = [
    "Apartment", "Loft", "House", "Townhouse", "Condominium", "Serviced apartment",
    "Hotel", "Hostel", "Guesthouse", "Bed and breakfast", "Boutique hotel"
]

df['Property Type Reduced'] = df['Property Type'].apply(
    lambda x: property_mapping.get(x, x) if x in top_property_types or x in property_mapping else "Other"
)

# 4. Binning Postal Codes (first two digits represent broad area)
df['Postal Code Reduced'] = df[df['Postal Code'].notna()]['Postal Code'].astype(str).str[:2]

## Transform/Manipulate data

In [40]:
# Extracting years from date columns -- will be moved to data proccessing stage
# df['Host Since'] = pd.to_datetime(df['Host Since'])
# df['Host Since Year'] = df['Host Since'].dt.year.astype('str').apply(lambda x: (x.split('.')[0]))
# df['Host Since Year'] = df['Host Since Year'].apply(lambda x: None if x == 'nan' else x).astype('category')

# df['Host Since Month'] = df['Host Since'].dt.month.astype('str').apply(lambda x: (x.split('.')[0]))
# df['Host Since Month'] = df['Host Since Month'].apply(lambda x: None if x == 'nan' else x).astype('category')

# df['Host Since Day'] = df['Host Since'].dt.day.astype('str').apply(lambda x: (x.split('.')[0]))
# df['Host Since Day'] = df['Host Since Day'].apply(lambda x: None if x == 'nan' else x).astype('category')

# 5. transform true/false into bool
df['Instant Bookable'] = df['Instant Bookable'].replace({'t': True, 'f': False})
df['Is Exact Location'] = df['Is Exact Location'].replace({'t': True, 'f': False})
df['Is Superhost'] = df['Is Superhost'].replace({'t': True, 'f': False})
df['Is Superhost'] = df['Is Superhost'].astype(bool)

## Pre-EDA (Exploratory Data Analysis)

Clean and drop redundant and duplicated columns before EDA

In [41]:
# https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-germany-postleitzahl/records?limit=20&refine=lan_name%3A%22Berlin%22

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23536 entries, 0 to 23535
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Accuracy Rating        18888 non-null  float64
 1   Bathrooms              23507 non-null  float64
 2   Checkin Rating         18870 non-null  float64
 3   Cleanliness Rating     18892 non-null  float64
 4   Communication Rating   18886 non-null  float64
 5   Host Response Rate     13046 non-null  float64
 6   Latitude               23536 non-null  float64
 7   Location Rating        18871 non-null  float64
 8   Longitude              23536 non-null  float64
 9   Overall Rating         18914 non-null  float64
 10  Price                  23536 non-null  float64
 11  Square Feet            425 non-null    float64
 12  Value Rating           18868 non-null  float64
 13  review_date            19380 non-null  object 
 14  Comments               19378 non-null  object 
 15  Ho

In [43]:
df_EDA = df.copy()
df_EDA.to_pickle("/content/drive/My Drive/Airbnb/df_EDA.pkl")
df_EDA.to_csv("/content/drive/My Drive/Airbnb/df_EDA.csv")

In [44]:
# cleaned_df = pd.read_pickle('/content/drive/My Drive/Airbnb/df_EDA.pkl')
# cleaned_df.info()